In [ ]:
# Dataset Partitioner for MNLP M3 DPO Training
# Creates train/validation/test splits with exact filtering logic from training script
# Uploads to HuggingFace or saves locally based on configuration

from transformers import AutoTokenizer
from datasets import load_dataset, concatenate_datasets, DatasetDict
import torch
import os
from huggingface_hub import login
import random

# Configuration matching your training script
CONFIG = {
    "base_model": "Qwen/Qwen3-0.6B-Base",
    "metamath_samples": 200000,
    "codealpaca_samples": 80000,
    "seed": 42,
    "max_len": 800,
    "max_prompt": 400,
    
    # EDIT THE LINE BELOW TO SET YOUR HUGGING FACE REPO:
    # Leave empty ("") to save locally only, or set to "username/dataset-name" to upload
    "hf_repo": "",  # Example: "username/my-dpo-dataset"
    
    # Split ratios (from total filtered samples)
    "test_ratio": 0.10,      # 10% for test
    "val_ratio": 0.10,       # 10% for validation  
    "train_ratio": 0.80,     # 80% for training
}

print("🚀 MNLP M3 DPO Dataset Partitioner")
print("=" * 50)
print(f"📊 Target samples: MetaMath={CONFIG['metamath_samples']:,}, CodeAlpaca={CONFIG['codealpaca_samples']:,}")
print(f"📈 Split ratios: Train={CONFIG['train_ratio']:.0%}, Val={CONFIG['val_ratio']:.0%}, Test={CONFIG['test_ratio']:.0%}")

# -------------------------------------------------------------------
# 1) Load tokenizer
print("\n📦 Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], use_fast=True)
tok.pad_token = tok.eos_token
tok.padding_side = "right"
tok.model_max_length = CONFIG["max_len"]

print(f"✅ Using fast tokenizer: {CONFIG['base_model']}")

# -------------------------------------------------------------------
# 2) Filtering logic
def keep_last_paragraph(example, index):
    """Apply only to MetaMath examples - keep last paragraph only and extract just the question"""
    if "prompt" in example and example["prompt"]:
        paragraphs = example["prompt"].strip().split("\n\n")
        last_paragraph = paragraphs[-1].strip()
        
        if "Question:" in last_paragraph:
            question_start = last_paragraph.find("Question:") + len("Question:")
            
            if "Answer:" in last_paragraph:
                answer_start = last_paragraph.find("Answer:")
                question_text = last_paragraph[question_start:answer_start].strip()
            else:
                question_text = last_paragraph[question_start:].strip()
            
            example["prompt"] = question_text
        else:
            example["prompt"] = last_paragraph
    
    example["dataset"] = "metamath"
    example["id"] = f"metamath{index}"
    
    if "system" in example:
        del example["system"]
    if "question" in example:
        del example["question"]
    
    return example

def process_codealpaca(example, index):
    """Process AlekseyKorshuk/evol-codealpaca-v1-dpo format to match our DPO format"""
    if "question" in example:
        example["prompt"] = example["question"]
    
    example["dataset"] = "codealpaca"
    example["id"] = f"codealpaca{index}"
    
    if "system" in example:
        del example["system"]
    if "question" in example:
        del example["question"]
    
    return example

def ok_lenient_fast(examples):
    """Fast CPU batch filtering - EXACT same logic as training script"""
    prompts = [ex.strip() if ex else "" for ex in examples["prompt"]]
    chosen = [ex.strip() if ex else "" for ex in examples["chosen"]]
    rejected = [ex.strip() if ex else "" for ex in examples["rejected"]]
    
    valid_mask = [bool(p and c and r) for p, c, r in zip(prompts, chosen, rejected)]
    
    if not any(valid_mask):
        return valid_mask
    
    try:
        prompt_tokens = tok(prompts, padding=False, truncation=False)["input_ids"]
        chosen_tokens = tok(chosen, padding=False, truncation=False)["input_ids"]
        rejected_tokens = tok(rejected, padding=False, truncation=False)["input_ids"]
        
        result = []
        for i, (p_ids, c_ids, r_ids) in enumerate(zip(prompt_tokens, chosen_tokens, rejected_tokens)):
            if not valid_mask[i]:
                result.append(False)
            else:
                p_len = len(p_ids)
                c_len = len(c_ids)
                r_len = len(r_ids)
                result.append(p_len <= CONFIG["max_prompt"] and p_len + max(c_len, r_len) <= CONFIG["max_len"])
        
        return result
        
    except Exception as e:
        print(f"⚠️ Batch filtering failed: {e}, using individual fallback")
        result = []
        for i, (p, c, r) in enumerate(zip(prompts, chosen, rejected)):
            if not valid_mask[i]:
                result.append(False)
            else:
                try:
                    p_len = len(tok(p)["input_ids"])
                    c_len = len(tok(c)["input_ids"])
                    r_len = len(tok(r)["input_ids"])
                    result.append(p_len <= CONFIG["max_prompt"] and p_len + max(c_len, r_len) <= CONFIG["max_len"])
                except:
                    result.append(False)
        return result

# -------------------------------------------------------------------
# 3) Create train/val/test splits
def create_three_way_split(dataset, test_ratio=0.1, val_ratio=0.1, seed=42):
    """Split dataset into train/val/test with specified ratios"""
    total_size = len(dataset)
    
    test_size = int(total_size * test_ratio)
    val_size = int(total_size * val_ratio)
    train_size = total_size - test_size - val_size
    
    print(f"    📊 Split sizes: Train={train_size:,}, Val={val_size:,}, Test={test_size:,}")
    
    dataset_shuffled = dataset.shuffle(seed=seed)
    
    test_split = dataset_shuffled.select(range(test_size))
    val_split = dataset_shuffled.select(range(test_size, test_size + val_size))
    train_split = dataset_shuffled.select(range(test_size + val_size, total_size))
    
    return train_split, val_split, test_split

# -------------------------------------------------------------------
# 4) Load and process MetaMath dataset
print("\n📚 Loading MetaMath dataset...")
print(f"🎯 Loading {CONFIG['metamath_samples']:,} samples to filter")

metamath_full = load_dataset("abacusai/MetaMath_DPO_FewShot", split="train")
print(f"📊 MetaMath full dataset: {len(metamath_full):,} samples")

initial_metamath_samples = min(CONFIG["metamath_samples"], len(metamath_full))
print(f"📥 Loading {initial_metamath_samples:,} samples for filtering...")

metamath_subset = metamath_full.shuffle(seed=CONFIG["seed"]).select(range(initial_metamath_samples))

print("🔍 Applying filtering and preprocessing...")
metamath_filtered = (metamath_subset
                    .map(keep_last_paragraph, with_indices=True)
                    .filter(ok_lenient_fast, batched=True, batch_size=2000))

print(f"✅ MetaMath after filtering: {len(metamath_filtered):,} samples")

print("📊 Creating MetaMath train/val/test splits...")
metamath_train, metamath_val, metamath_test = create_three_way_split(
    metamath_filtered, 
    test_ratio=CONFIG["test_ratio"], 
    val_ratio=CONFIG["val_ratio"], 
    seed=CONFIG["seed"]
)

# -------------------------------------------------------------------
# 5) Load and process EvolCodeAlpaca dataset
print("\n📚 Loading EvolCodeAlpaca dataset...")
print(f"🎯 Loading {CONFIG['codealpaca_samples']:,} samples to filter")

codealpaca_full = load_dataset("AlekseyKorshuk/evol-codealpaca-v1-dpo", split="train")
print(f"📊 EvolCodeAlpaca full dataset: {len(codealpaca_full):,} samples")

initial_codealpaca_samples = min(CONFIG["codealpaca_samples"], len(codealpaca_full))
print(f"📥 Loading {initial_codealpaca_samples:,} samples for filtering...")

codealpaca_subset = codealpaca_full.shuffle(seed=CONFIG["seed"]).select(range(initial_codealpaca_samples))

print("🔍 Applying filtering and preprocessing...")
codealpaca_filtered = (codealpaca_subset
                      .map(process_codealpaca, with_indices=True)
                      .filter(ok_lenient_fast, batched=True, batch_size=2000))

print(f"✅ EvolCodeAlpaca after filtering: {len(codealpaca_filtered):,} samples")

print("📊 Creating EvolCodeAlpaca train/val/test splits...")
codealpaca_train, codealpaca_val, codealpaca_test = create_three_way_split(
    codealpaca_filtered,
    test_ratio=CONFIG["test_ratio"],
    val_ratio=CONFIG["val_ratio"], 
    seed=CONFIG["seed"]
)

# -------------------------------------------------------------------
# 6) Combine datasets
print("\n🔗 Combining datasets...")

combined_train = concatenate_datasets([metamath_train, codealpaca_train]).shuffle(seed=CONFIG["seed"])
print(f"✅ Combined train: {len(combined_train):,} samples")

combined_val = concatenate_datasets([metamath_val, codealpaca_val]).shuffle(seed=CONFIG["seed"])
print(f"✅ Combined validation: {len(combined_val):,} samples")

combined_test = concatenate_datasets([metamath_test, codealpaca_test]).shuffle(seed=CONFIG["seed"])
print(f"✅ Combined test: {len(combined_test):,} samples")

final_dataset = DatasetDict({
    "train": combined_train,
    "validation": combined_val,
    "test": combined_test
})

# -------------------------------------------------------------------
# 7) Dataset statistics and validation
print("\n📊 FINAL DATASET STATISTICS")
print("=" * 50)

total_samples = len(combined_train) + len(combined_val) + len(combined_test)
train_pct = len(combined_train) / total_samples * 100
val_pct = len(combined_val) / total_samples * 100
test_pct = len(combined_test) / total_samples * 100

print(f"Train:      {len(combined_train):,} samples ({train_pct:.1f}%)")
print(f"Validation: {len(combined_val):,} samples ({val_pct:.1f}%)")
print(f"Test:       {len(combined_test):,} samples ({test_pct:.1f}%)")
print(f"Total:      {total_samples:,} samples")

metamath_total = len(metamath_train) + len(metamath_val) + len(metamath_test)
codealpaca_total = len(codealpaca_train) + len(codealpaca_val) + len(codealpaca_test)

print(f"\n📈 Dataset Composition:")
print(f"MetaMath:      {metamath_total:,} samples ({metamath_total/total_samples*100:.1f}%)")
print(f"EvolCodeAlpaca: {codealpaca_total:,} samples ({codealpaca_total/total_samples*100:.1f}%)")

print(f"\n🔍 Sample Validation:")
sample_train = combined_train[0]
sample_val = combined_val[0] 
sample_test = combined_test[0]

print(f"Train sample keys: {list(sample_train.keys())}")
print(f"Dataset: {sample_train.get('dataset', 'N/A')}")
print(f"ID: {sample_train.get('id', 'N/A')}")
print(f"Prompt length: {len(sample_train['prompt'])} chars")
print(f"Chosen length: {len(sample_train['chosen'])} chars")
print(f"Rejected length: {len(sample_train['rejected'])} chars")

# -------------------------------------------------------------------
# 8) Save locally or upload to Hugging Face
if CONFIG["hf_repo"]:
    print(f"\n🚀 Uploading to Hugging Face: {CONFIG['hf_repo']}")
    
    try:
        print("🔐 Logging into Hugging Face...")
        print("⚠️  Make sure you have set your HF_TOKEN environment variable or run 'huggingface-cli login'")
        
        hf_token = os.environ.get("HF_TOKEN")
        if hf_token:
            login(token=hf_token)
            print("✅ Logged in using HF_TOKEN environment variable")
        else:
            print("⚠️  No HF_TOKEN found, assuming you're already logged in")
        
        dataset_card = f"""
# MNLP M3 DPO Dataset

This dataset contains {total_samples:,} samples for Direct Preference Optimization (DPO) training, combining mathematical reasoning and code generation tasks.

## Dataset Composition

- **MetaMath**: {metamath_total:,} samples ({metamath_total/total_samples*100:.1f}%) - Mathematical reasoning with DPO pairs
- **EvolCodeAlpaca**: {codealpaca_total:,} samples ({codealpaca_total/total_samples*100:.1f}%) - Code generation with DPO pairs

## Dataset Splits

- **Train**: {len(combined_train):,} samples ({train_pct:.1f}%)
- **Validation**: {len(combined_val):,} samples ({val_pct:.1f}%)
- **Test**: {len(combined_test):,} samples ({test_pct:.1f}%)

## Dataset Schema

Each sample contains the following fields:

- **prompt**: The input question/problem
  - For MetaMath: Mathematical problems (last paragraph only)
  - For EvolCodeAlpaca: Programming questions (question field only, no system message)
- **chosen**: The preferred response
- **rejected**: The less preferred response  
- **dataset**: Source dataset identifier ("metamath" or "codealpaca")
- **id**: Unique identifier (e.g., "metamath0", "metamath1", "codealpaca0", "codealpaca1")

## Filtering and Processing

The dataset has been processed with the following steps:

1. **Token Length Filtering**: 
   - Max prompt length: {CONFIG['max_prompt']} tokens
   - Max total length: {CONFIG['max_len']} tokens

2. **MetaMath Processing**:
   - Applied `keep_last_paragraph()` to get the final question from multi-question prompts
   - **Extracted clean questions**: Removed "Question:" prefix and "Answer:" suffix from last paragraph
   - Added dataset="metamath" and sequential IDs
   - Deleted system and question columns if present
   - Filtered for quality and length constraints

3. **EvolCodeAlpaca Processing**:
   - Used only the 'question' field as prompt (removed system messages)
   - Added dataset="codealpaca" and sequential IDs
   - Deleted system and question columns
   - Applied same length and quality filters

## Usage

```python
from datasets import load_dataset

dataset = load_dataset("{CONFIG['hf_repo']}")

# Access splits
train_data = dataset["train"]
val_data = dataset["validation"] 
test_data = dataset["test"]
```

## Source Datasets

- [MetaMath DPO FewShot](https://huggingface.co/datasets/abacusai/MetaMath_DPO_FewShot)
- [EvolCodeAlpaca v1 DPO](https://huggingface.co/datasets/AlekseyKorshuk/evol-codealpaca-v1-dpo)

## Citation

If you use this dataset, please cite the original source datasets and this preparation:

```bibtex
@misc{{mnlp_m3_dpo_dataset,
  title={{MNLP M3 DPO Dataset}},
  author={{Albert Fares}},
  year={{2025}},
  publisher={{Hugging Face}},
  url={{https://huggingface.co/datasets/{CONFIG['hf_repo']}}}
}}
```
"""

        print("📤 Uploading dataset...")
        final_dataset.push_to_hub(
            CONFIG["hf_repo"],
            private=False,  # Change to True if you want private dataset
            token=hf_token if hf_token else None
        )
        
        from huggingface_hub import HfApi
        api = HfApi()
        
        print("📝 Uploading dataset card...")
        api.upload_file(
            path_or_fileobj=dataset_card.encode(),
            path_in_repo="README.md",
            repo_id=CONFIG["hf_repo"],
            repo_type="dataset",
            token=hf_token if hf_token else None
        )
        
        print("✅ Successfully uploaded to Hugging Face!")
        print(f"🌐 Dataset URL: https://huggingface.co/datasets/{CONFIG['hf_repo']}")
        
    except Exception as e:
        print(f"❌ Error uploading to Hugging Face: {e}")
        print("💾 Saving locally instead...")
        
        # Save locally as backup
        local_path = "./mnlp_m3_dpo_dataset"
        final_dataset.save_to_disk(local_path)
        print(f"✅ Dataset saved locally to: {local_path}")

else:
    print("\n💾 No HF repo specified - saving locally only...")
    local_path = "./mnlp_m3_dpo_dataset"
    final_dataset.save_to_disk(local_path)
    print(f"✅ Dataset saved locally to: {local_path}")

# -------------------------------------------------------------------
print("\n" + "="*60)
print("📋 DATASET CREATION SUMMARY")
print("="*60)

print(f"\n🎯 Successfully created MNLP M3 DPO dataset:")
print(f"   • Total samples: {total_samples:,}")
print(f"   • Train/Val/Test: {len(combined_train):,}/{len(combined_val):,}/{len(combined_test):,}")
print(f"   • Math/Code ratio: {metamath_total:,}/{codealpaca_total:,}")

print(f"\n📊 Sample Examples:")
print("="*40)

print(f"\n🧮 MetaMath Example:")
metamath_sample = next((ex for ex in combined_train if ex.get('dataset') == 'metamath'), None)
if metamath_sample:
    print(f"ID: {metamath_sample['id']}")
    print(f"Dataset: {metamath_sample['dataset']}")
    print(f"Prompt: {metamath_sample['prompt'][:100]}...")
    print(f"Chosen: {metamath_sample['chosen'][:100]}...")

print(f"\n💻 EvolCodeAlpaca Example:")
code_sample = next((ex for ex in combined_train if ex.get('dataset') == 'codealpaca'), None)
if code_sample:
    print(f"ID: {code_sample['id']}")
    print(f"Dataset: {code_sample['dataset']}")
    print(f"Prompt: {code_sample['prompt'][:100]}...")
    print(f"Chosen: {code_sample['chosen'][:100]}...")

print(f"\n🚀 Dataset ready for DPO training!")
if CONFIG["hf_repo"]:
    print(f"📍 Location: {CONFIG['hf_repo']}")
else:
    print(f"📍 Location: ./mnlp_m3_dpo_dataset")
print("="*60)